## Crop Yield Prediction: Baseline and Core Models

This notebook uses the merged features in `data/processed/crop_yield_ml_features_with_ndvi.csv` (including NDVI) to train and compare multiple models:

- Linear Regression (baseline)
- Decision Tree
- Random Forest
- XGBoost (if `xgboost` is installed)
- LightGBM (if `lightgbm` is installed)
- Stacking Ensemble

Metrics:
- R² Score
- RMSE
- MAE
- 5-fold cross-validation estimates for all metrics


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.svm import SVR

try:
    from xgboost import XGBRegressor
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False
    print("xgboost not installed; XGBoost model will be skipped.")

try:
    from lightgbm import LGBMRegressor
    LGBM_AVAILABLE = True
except ImportError:
    LGBM_AVAILABLE = False
    print("lightgbm not installed; LightGBM model will be skipped.")

pd.set_option("display.max_columns", None)


In [ ]:
# 1. Load merged feature dataset (with NDVI)
data_path = "../data/processed/crop_yield_ml_features_with_ndvi.csv"
df = pd.read_csv(data_path)
print("Shape before dropna:", df.shape)
df = df.dropna(subset=["mean_ndvi"])
print("Shape after dropping rows with missing NDVI:", df.shape)
df.head()

In [ ]:
# 2. Define features and target
target_col = "yield"
y = df[target_col].values

# Exclude 'production' to avoid target leakage (yield ≈ production/area)
feature_cols = [
    "crop", "season", "state",  # categorical
    "year", "area", "fertilizer", "pesticide",
    "N", "P", "K", "pH",
    "avg_temp_c", "total_rainfall_mm", "avg_humidity_percent",
    "mean_ndvi",  # NDVI feature
]
X = df[feature_cols].copy()

categorical_cols = ["crop", "season", "state"]
numeric_cols = [c for c in feature_cols if c not in categorical_cols]

print("Categorical features:", categorical_cols)
print("Numeric features:", numeric_cols)

In [ ]:
# 3. Train/validation split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_train.shape, X_test.shape

In [ ]:
# 4. Preprocessing: one-hot encode categoricals, scale numerics
categorical_transformer = OneHotEncoder(handle_unknown="ignore")
numeric_transformer = StandardScaler()

preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", categorical_transformer, categorical_cols),
        ("numeric", numeric_transformer, numeric_cols),
    ]
)
preprocessor

In [ ]:
# Helper function to evaluate models
def evaluate_model(name, model):
    print(f"\n=== {name} ===")
    pipe = Pipeline(steps=[("preprocess", preprocessor), ("model", model)])

    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    scoring = {
        "r2": "r2",
        "rmse": "neg_root_mean_squared_error",
        "mae": "neg_mean_absolute_error",
    }

    cv_results = cross_validate(
        pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1, return_train_score=False
    )

    # Fit on full training set and evaluate on hold-out test set
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)

    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)

    summary = {
        "model": name,
        "test_r2": r2,
        "test_rmse": rmse,
        "test_mae": mae,
        "cv_r2_mean": cv_results["test_r2"].mean(),
        "cv_r2_std": cv_results["test_r2"].std(),
        "cv_rmse_mean": -cv_results["test_rmse"].mean(),  # invert sign
        "cv_rmse_std": cv_results["test_rmse"].std(),
        "cv_mae_mean": -cv_results["test_mae"].mean(),
        "cv_mae_std": cv_results["test_mae"].std(),
    }

    print("Test R2:", r2)
    print("Test RMSE:", rmse)
    print("Test MAE:", mae)
    print("CV R2 (mean ± std):", summary["cv_r2_mean"], "+/-", summary["cv_r2_std"])
    print("CV RMSE (mean):", summary["cv_rmse_mean"])
    print("CV MAE (mean):", summary["cv_mae_mean"])

    return summary


In [ ]:
# 5. Define models
models = []

# 1. Baseline: Linear Regression
models.append(("Linear Regression", LinearRegression()))

# 2. Core models
models.append(("Decision Tree", DecisionTreeRegressor(random_state=42)))
models.append(("Random Forest", RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)))

if XGB_AVAILABLE:
    models.append((
        "XGBoost",
        XGBRegressor(
            n_estimators=400,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="reg:squarederror",
            n_jobs=-1,
            random_state=42,
        ),
    ))

models.append(("SVR (RBF)", SVR(kernel="rbf", C=10.0, epsilon=0.1)))

# 3. Advanced: LightGBM (if available)
if LGBM_AVAILABLE:
    models.append((
        "LightGBM",
        LGBMRegressor(
            n_estimators=400,
            learning_rate=0.05,
            num_leaves=31,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
        ),
    ))

models

In [ ]:
# 6. Run all models and collect results
results = []
for name, mdl in models:
    summary = evaluate_model(name, mdl)
    results.append(summary)

results_df = pd.DataFrame(results)
results_df_sorted = results_df.sort_values(by="test_r2", ascending=False)
results_df_sorted

In [ ]:
# 7. Stacking Ensemble (advanced)
base_estimators = []
for name, mdl in models:
    if name in ["Linear Regression"]:
        continue
    base_estimators.append((name.replace(" ", "_"), mdl))

final_estimator = LinearRegression()

stack_model = StackingRegressor(
    estimators=base_estimators,
    final_estimator=final_estimator,
    n_jobs=-1,
)

stack_summary = evaluate_model("Stacking Ensemble", stack_model)
results.append(stack_summary)
results_full = pd.DataFrame(results).sort_values(by="test_r2", ascending=False)
results_full

In [ ]:
# 8. Visual comparison of model performance
plt.figure(figsize=(10, 5))
sns.barplot(
    data=results_full,
    x="model",
    y="test_r2",
    order=results_full.sort_values("test_r2", ascending=False)["model"],
)
plt.xticks(rotation=45, ha="right")
plt.title("Model comparison (Test R²)")
plt.tight_layout()
plt.show()

results_full